In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os, math, warnings, pickle, time
import pandas as pd
import numpy as np
warnings.filterwarnings('ignore')

PROJECT = "/content/drive/MyDrive/delivery_delay_project"
MODELS  = f"{PROJECT}/models"
RAW     = f"{PROJECT}/data/raw"

print("✅ Ready!")

Mounted at /content/drive
✅ Ready!


In [ ]:
from sklearn.preprocessing import LabelEncoder

# Load XGBoost (best model)
with open(f"{MODELS}/xgboost_model.pkl",'rb') as f:
    xgb_model = pickle.load(f)

print("✅ Model loaded!")

FEATURES = ['Delivery_person_Age','Delivery_person_Ratings',
            'distance_km','order_hour','is_peak_hour',
            'rain_flag','festival_flag','multi_delivery',
            'Vehicle_condition','traffic_enc','weather_enc',
            'city_enc','vehicle_enc']

✅ Model loaded!


In [ ]:
def predict_eta(
    age, rating, distance_km,
    order_hour, rain=0, festival=0,
    multi=0, vehicle_condition=2,
    traffic='Medium', weather='Clear',
    city='Metropolitian'
):
    # Encode categoricals manually
    traffic_map = {'Low':0, 'Medium':1, 'High':2, 'Jam':3}
    weather_map = {'Cloudy':0, 'Fog':1, 'Sandstorms':2,
                   'Stormy':3, 'Sunny':4, 'Windy':5, 'Clear':4}
    city_map    = {'Metropolitian':0, 'Semi-Urban':1, 'Urban':2}

    features = pd.DataFrame([{
        'Delivery_person_Age'     : age,
        'Delivery_person_Ratings' : rating,
        'distance_km'             : distance_km,
        'order_hour'              : order_hour,
        'is_peak_hour'            : 1 if order_hour in list(range(12,14))+list(range(19,22)) else 0,
        'rain_flag'               : rain,
        'festival_flag'           : festival,
        'multi_delivery'          : multi,
        'Vehicle_condition'       : vehicle_condition,
        'traffic_enc'             : traffic_map.get(traffic, 1),
        'weather_enc'             : weather_map.get(weather, 4),
        'city_enc'                : city_map.get(city, 0),
        'vehicle_enc'             : 0,
    }])

    eta = xgb_model.predict(features[FEATURES])[0]
    return round(float(eta), 1)

print("✅ Prediction function ready!")

✅ Prediction function ready!


In [ ]:
scenarios = [
    {
        "name"      : "Normal Day — Short Distance",
        "age":25, "rating":4.5, "distance_km":2.5,
        "order_hour":14, "rain":0, "festival":0,
        "traffic":"Low", "weather":"Sunny", "city":"Metropolitian"
    },
    {
        "name"      : "Peak Hour — Heavy Traffic",
        "age":29, "rating":4.2, "distance_km":5.0,
        "order_hour":20, "rain":0, "festival":0,
        "traffic":"High", "weather":"Clear", "city":"Metropolitian"
    },
    {
        "name"      : "Rainy Evening — Mumbai",
        "age":27, "rating":4.0, "distance_km":4.5,
        "order_hour":19, "rain":1, "festival":0,
        "traffic":"Jam", "weather":"Stormy", "city":"Metropolitian"
    },
    {
        "name"      : "Diwali Festival — High Demand",
        "age":30, "rating":3.8, "distance_km":6.0,
        "order_hour":21, "rain":0, "festival":1,
        "traffic":"Jam", "weather":"Cloudy", "city":"Metropolitian"
    },
    {
        "name"      : "Early Morning — Best Case",
        "age":24, "rating":4.9, "distance_km":1.5,
        "order_hour":8, "rain":0, "festival":0,
        "traffic":"Low", "weather":"Sunny", "city":"Urban"
    },
]

print("━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━")
print("🛵 DELIVERY ETA PREDICTIONS — SCENARIO ANALYSIS")
print("━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━")

results = []
for s in scenarios:
    eta = predict_eta(
        age=s['age'], rating=s['rating'],
        distance_km=s['distance_km'],
        order_hour=s['order_hour'],
        rain=s.get('rain',0),
        festival=s.get('festival',0),
        traffic=s['traffic'],
        weather=s['weather'],
        city=s['city']
    )
    status = "🔴 DELAY RISK" if eta > 35 else "🟡 MODERATE" if eta > 25 else "🟢 ON TIME"
    print(f"\n  📍 {s['name']}")
    print(f"     Distance : {s['distance_km']} km | Hour: {s['order_hour']}:00")
    print(f"     Weather  : {s['weather']} | Traffic: {s['traffic']}")
    print(f"     ⏱️  ETA   : {eta} min  {status}")
    results.append({'Scenario': s['name'], 'ETA_min': eta, 'Status': status})

print("\n━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━")

# Save for Power BI
pd.DataFrame(results).to_csv(f"{PROJECT}/scenario_predictions.csv", index=False)
print("✅ Saved: scenario_predictions.csv")

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
🛵 DELIVERY ETA PREDICTIONS — SCENARIO ANALYSIS
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

  📍 Normal Day — Short Distance
     Distance : 2.5 km | Hour: 14:00
     Weather  : Sunny | Traffic: Low
     ⏱️  ETA   : 19.3 min  🟢 ON TIME

  📍 Peak Hour — Heavy Traffic
     Distance : 5.0 km | Hour: 20:00
     Weather  : Clear | Traffic: High
     ⏱️  ETA   : 22.5 min  🟢 ON TIME

  📍 Rainy Evening — Mumbai
     Distance : 4.5 km | Hour: 19:00
     Weather  : Stormy | Traffic: Jam
     ⏱️  ETA   : 29.8 min  🟡 MODERATE

  📍 Diwali Festival — High Demand
     Distance : 6.0 km | Hour: 21:00
     Weather  : Cloudy | Traffic: Jam
     ⏱️  ETA   : 31.4 min  🟡 MODERATE

  📍 Early Morning — Best Case
     Distance : 1.5 km | Hour: 8:00
     Weather  : Sunny | Traffic: Low
     ⏱️  ETA   : 19.4 min  🟢 ON TIME

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
✅ Saved: scenario_predictions.csv


In [ ]:
import time

print("🔄 DYNAMIC RE-PREDICTION ENGINE")
print("   Simulating real-time ETA updates as conditions change\n")
print("━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━")

# Simulate an active order over 10 minutes
# Conditions change as order progresses
order_states = [
    {"minute":0,  "distance_km":5.0, "rain":0, "traffic":"Medium", "note":"Order placed"},
    {"minute":2,  "distance_km":4.2, "rain":0, "traffic":"High",   "note":"Traffic building up"},
    {"minute":4,  "distance_km":3.1, "rain":1, "traffic":"High",   "note":"Rain started 🌧️"},
    {"minute":6,  "distance_km":2.0, "rain":1, "traffic":"Jam",    "note":"Traffic jam detected"},
    {"minute":8,  "distance_km":1.1, "rain":1, "traffic":"Medium", "note":"Jam clearing"},
    {"minute":10, "distance_km":0.3, "rain":0, "traffic":"Low",    "note":"Almost there!"},
]

log = []
for state in order_states:
    eta = predict_eta(
        age=26, rating=4.3,
        distance_km=state['distance_km'],
        order_hour=20,
        rain=state['rain'],
        traffic=state['traffic'],
        city='Metropolitian'
    )
    remaining = max(0, eta - state['minute'])
    alert = " ⚠️ DELAY ALERT!" if remaining > 25 else ""

    print(f"  ⏰ T+{state['minute']:02d} min | Remaining: {state['distance_km']} km")
    print(f"         Conditions : {state['traffic']} traffic | Rain: {'Yes' if state['rain'] else 'No'}")
    print(f"         Updated ETA: {remaining:.0f} min remaining  {alert}")
    print(f"         Note       : {state['note']}\n")

    log.append({
        'minute'      : state['minute'],
        'distance_km' : state['distance_km'],
        'predicted_eta': remaining,
        'rain'        : state['rain'],
        'traffic'     : state['traffic'],
        'note'        : state['note']
    })

# Save log for Power BI
pd.DataFrame(log).to_csv(f"{PROJECT}/reprediction_log.csv", index=False)
print("━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━")
print("✅ Saved: reprediction_log.csv")
print("   This CSV powers your Power BI 'Live Tracking' visual!")

🔄 DYNAMIC RE-PREDICTION ENGINE
   Simulating real-time ETA updates as conditions change

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
  ⏰ T+00 min | Remaining: 5.0 km
         Conditions : Medium traffic | Rain: No
         Updated ETA: 30 min remaining   ⚠️ DELAY ALERT!
         Note       : Order placed

  ⏰ T+02 min | Remaining: 4.2 km
         Conditions : High traffic | Rain: No
         Updated ETA: 20 min remaining  
         Note       : Traffic building up

  ⏰ T+04 min | Remaining: 3.1 km
         Conditions : High traffic | Rain: Yes
         Updated ETA: 18 min remaining  
         Note       : Rain started 🌧️

  ⏰ T+06 min | Remaining: 2.0 km
         Conditions : Jam traffic | Rain: Yes
         Updated ETA: 20 min remaining  
         Note       : Traffic jam detected

  ⏰ T+08 min | Remaining: 1.1 km
         Conditions : Medium traffic | Rain: Yes
         Updated ETA: 19 min remaining  
         Note       : Jam clearing

  ⏰ T+10 min | Remaining: 0.3 km
      

In [ ]:
import math
from sklearn.preprocessing import LabelEncoder

# Reload full dataset with predictions
df = pd.read_csv(f"{RAW}/train.csv")

# Convert problematic columns to numeric
df['Delivery_person_Age'] = pd.to_numeric(df['Delivery_person_Age'], errors='coerce')
df['Delivery_person_Age'].fillna(df['Delivery_person_Age'].median(), inplace=True)
df['Delivery_person_Ratings'] = pd.to_numeric(df['Delivery_person_Ratings'], errors='coerce')
df['Delivery_person_Ratings'].fillna(df['Delivery_person_Ratings'].median(), inplace=True)

time_col        = [c for c in df.columns if 'time_taken' in c.lower()][0]
df['time_taken']= df[time_col].astype(str).str.strip()\
                    .str.replace(r'[^0-9.]','',regex=True)\
                    .replace('',np.nan).astype(float)
weather_col     = [c for c in df.columns if 'weather' in c.lower()][0]
df['weather']   = df[weather_col].astype(str)\
                    .str.replace("conditions ","",regex=False).str.strip()

def haversine(lat1,lon1,lat2,lon2):
    R=6371
    lat1,lon1,lat2,lon2=map(math.radians,[lat1,lon1,lat2,lon2])
    dlat=lat2-lat1; dlon=lon2-lon1
    a=math.sin(dlat/2)**2+math.cos(lat1)*math.cos(lat2)*math.sin(dlon/2)**2
    return 2*R*math.asin(math.sqrt(a))

df['distance_km']   = df.apply(lambda r: haversine(
    r['Restaurant_latitude'],r['Restaurant_longitude'],
    r['Delivery_location_latitude'],r['Delivery_location_longitude']),axis=1)

time_order_col      = [c for c in df.columns if 'time_orderd' in c.lower()][0]
df['order_hour']    = pd.to_datetime(df[time_order_col],
                        errors='coerce').dt.hour.fillna(12).astype(int)
df['is_peak_hour']  = df['order_hour'].apply(
                        lambda h: 1 if h in list(range(12,14))+list(range(19,22)) else 0)
df['rain_flag']     = df['weather'].str.lower().str.contains('rain',na=False).astype(int)
df['festival_flag'] = (df['Festival'].str.strip().str.lower()=='yes').astype(int)
df['multi_delivery']= pd.to_numeric(df['multiple_deliveries'].astype(str)\
                        .str.strip(),errors='coerce').fillna(0).astype(int)

le = LabelEncoder()
df['traffic_enc']   = le.fit_transform(df['Road_traffic_density'].astype(str))
df['weather_enc']   = le.fit_transform(df['weather'].astype(str))
df['city_enc']      = le.fit_transform(df['City'].astype(str))
df['vehicle_enc']   = le.fit_transform(df['Type_of_vehicle'].astype(str))

FEATURES = ['Delivery_person_Age','Delivery_person_Ratings',
            'distance_km','order_hour','is_peak_hour',
            'rain_flag','festival_flag','multi_delivery',
            'Vehicle_condition','traffic_enc','weather_enc',
            'city_enc','vehicle_enc']

# Add predictions to full dataset
df['predicted_time'] = xgb_model.predict(df[FEATURES])
df['delay_minutes']  = df['time_taken'] - df['predicted_time']
df['is_delayed']     = (df['delay_minutes'] > 5).astype(int)
df['on_time']        = (df['is_delayed'] == 0).astype(int)
df['delay_risk']     = pd.cut(df['predicted_time'],
                        bins=[0,20,30,40,100],
                        labels=['Low','Medium','High','Critical'])

# Save master file for Power BI
output_cols = [
    'Delivery_person_Age','Delivery_person_Ratings',
    'distance_km','order_hour','is_peak_hour',
    'rain_flag','festival_flag','multi_delivery',
    'Road_traffic_density','weather','City',
    'Type_of_vehicle','Vehicle_condition',
    'time_taken','predicted_time',
    'delay_minutes','is_delayed','on_time','delay_risk'
]

df[output_cols].to_csv(f"{PROJECT}/powerbi_master.csv", index=False)

print("✅ Power BI master file saved!")
print(f"   Rows          : {len(df)}")
print(f"   On-time %     : {df['on_time'].mean()*100:.1f}%")
print(f"   Avg delay     : {df['delay_minutes'].mean():.1f} min")
print(f"   Delayed orders: {df['is_delayed'].sum():,}")
print(f"\n📁 Files ready for Power BI:")
print(f"   ✅ powerbi_master.csv       ← main dataset with predictions")
print(f"   ✅ scenario_predictions.csv ← scenario comparison")
print(f"   ✅ reprediction_log.csv     ← live tracking simulation")
print(f"   ✅ shap_feature_importance.csv ← feature importance chart")

✅ Power BI master file saved!
   Rows          : 45593
   On-time %     : 90.6%
   Avg delay     : -0.0 min
   Delayed orders: 4,275

📁 Files ready for Power BI:
   ✅ powerbi_master.csv       ← main dataset with predictions
   ✅ scenario_predictions.csv ← scenario comparison
   ✅ reprediction_log.csv     ← live tracking simulation
   ✅ shap_feature_importance.csv ← feature importance chart
